# Assess evidence for future clinical reasoning tasks

Count unique patients meeting each versioned evidence definition. Task decisions retain the occurrence IDs a researcher can use to audit eligibility.

Select the dataset through the Compose environment or set `MEDLENS_DATASET` before opening the kernel. The resolved configuration determines the enabled tables and policies.

In [ ]:
%matplotlib inline
from medlens.notebook import workspace, notebook_run
import matplotlib.pyplot as plt

research = workspace()

The run records its input versions, resolved configuration, and producing stages. Set `MEDLENS_PARENT` to replay from a specific published manifest. The default walkthrough uses the preceding notebook checkpoint for this configuration.

In [ ]:
run = notebook_run(research, 'readiness')
print("Run", run["run_id"], "Parent", run["parent"])

In [ ]:
stage = next(s for s in research.config.stages.values() if s.enabled and s.op == "readiness")
report = run["tables"][stage.output]["stats"]
if "batch" in report:
    print("Readiness counts for this input batch")
report = report.get("batch", report)
print("Source patient rows", report["population_rows"], "Identifiable patients", report["unique_identifiable_patients"])
tasks = sorted(research.config.tasks)
counts = {(r["task"], r["status"]): r["count"] for r in report["tasks"]}
left = [0] * len(tasks)
for status in ("eligible", "ineligible", "indeterminate"):
    values = [counts.get((task, status), 0) for task in tasks]
    plt.barh(tasks, values, left=left, label=status)
    left = [a + b for a, b in zip(left, values)]
plt.xlabel("Patients")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()
research.frame(run, stage.output).select("task","status","criterion_counts").orderBy("task","status").show(5)

In [ ]:
research.close()